In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load cleaned dataset
df = pd.read_csv((r"C:\Users\warmi\GitHub\cardekho\experiments\cardekho_clean.csv"))

# Define features and target
numeric = ["age", "km_driven", "mileage(km/ltr/kg)", "engine", "max_power", "seats"]
categorical = ["fuel", "seller_type", "transmission", "owner", "brand"]
X = df[numeric + categorical]
y = df["selling_price"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Preprocessor (same for all models)
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical)
])

## Ridge Regression 

In [8]:
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error

# Ridge model
ridge = Ridge(alpha=1.0, random_state=42)
pipeline = Pipeline([("pre", preprocessor), ("model", ridge)])
pipeline.fit(X_train, y_train)
preds = pipeline.predict(X_test)

# Metrics (using new root_mean_squared_error)
ridge_scores = {
    "Model": "Ridge",
    "MAE": mean_absolute_error(y_test, preds),
    "RMSE": root_mean_squared_error(y_test, preds),  # new function
    "R²": r2_score(y_test, preds)
}

ridge_scores

{'Model': 'Ridge',
 'MAE': 163118.62448237833,
 'RMSE': 317533.30725268176,
 'R²': 0.8461788030352343}

## Random Forest

In [9]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error

rf = RandomForestRegressor(n_estimators=200, random_state=42)
pipeline = Pipeline([("pre", preprocessor), ("model", rf)])
pipeline.fit(X_train, y_train)
preds = pipeline.predict(X_test)

rf_scores = {
    "Model": "Random Forest",
    "MAE": mean_absolute_error(y_test, preds),
    "RMSE": root_mean_squared_error(y_test, preds),
    "R²": r2_score(y_test, preds)
}
rf_scores

{'Model': 'Random Forest',
 'MAE': 67410.03223584336,
 'RMSE': 139526.81127091762,
 'R²': 0.9703002392376977}

## XGBoost

In [10]:
from xgboost import XGBRegressor

xgb = XGBRegressor(n_estimators=300, learning_rate=0.05, random_state=42)
pipeline = Pipeline([("pre", preprocessor), ("model", xgb)])
pipeline.fit(X_train, y_train)
preds = pipeline.predict(X_test)

xgb_scores = {
    "Model": "XGBoost",
    "MAE": mean_absolute_error(y_test, preds),
    "RMSE": root_mean_squared_error(y_test, preds),
    "R²": r2_score(y_test, preds)
}
xgb_scores

{'Model': 'XGBoost',
 'MAE': 69075.0859375,
 'RMSE': 143243.921875,
 'R²': 0.9686967134475708}

## LightGBM

In [12]:
from lightgbm import LGBMRegressor

lgbm = LGBMRegressor(n_estimators=300, learning_rate=0.05, random_state=42)
pipeline = Pipeline([("pre", preprocessor), ("model", lgbm)])
pipeline.fit(X_train, y_train)
preds = pipeline.predict(X_test)

lgbm_scores = {
    "Model": "LightGBM",
    "MAE": mean_absolute_error(y_test, preds),
    "RMSE": root_mean_squared_error(y_test, preds),
    "R²": r2_score(y_test, preds)
}
lgbm_scores

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002458 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 924
[LightGBM] [Info] Number of data points in the train set: 6502, number of used features: 40
[LightGBM] [Info] Start training from score 638656.852815


c:\Users\warmi\GitHub\cardekho\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


{'Model': 'LightGBM',
 'MAE': 71143.01215947812,
 'RMSE': 142307.87676067158,
 'R²': 0.9691044812968741}

## CatBoost

In [13]:
from catboost import CatBoostRegressor

cat = CatBoostRegressor(iterations=300, learning_rate=0.05, depth=8, random_state=42, verbose=0)
pipeline = Pipeline([("pre", preprocessor), ("model", cat)])
pipeline.fit(X_train, y_train)
preds = pipeline.predict(X_test)

cat_scores = {
    "Model": "CatBoost",
    "MAE": mean_absolute_error(y_test, preds),
    "RMSE": root_mean_squared_error(y_test, preds),
    "R²": r2_score(y_test, preds)
}
cat_scores

{'Model': 'CatBoost',
 'MAE': 74721.228991478,
 'RMSE': 139521.53573824881,
 'R²': 0.9703024851013371}

## Comparison

In [15]:
results = pd.DataFrame([ridge_scores, rf_scores, xgb_scores, lgbm_scores, cat_scores])
results

,Model,MAE,RMSE,R²
0,Ridge,163118.624482,317533.307253,0.846179
1,Random Forest,67410.032236,139526.811271,0.970300
2,XGBoost,69075.085938,143243.921875,0.968697
3,LightGBM,71143.012159,142307.876761,0.969104
4,CatBoost,74721.228991,139521.535738,0.970302
